# Beauty and Personal Care — Source and Dataset Audit

This notebook visually inspects the fixed Amazon Reviews 2023 Beauty snapshot and validates the versioned canonical review artifact.

## Objectives

- verify the registered source files and dataset identity;
- inspect representative review and product metadata records;
- distinguish sample observations from exact full-population statistics;
- validate preprocessing reconciliation and canonical schema;
- document data limitations before catalog and ML work.

# Аудит источников и датасета

Этот ноутбук позволяет глазами проверить зафиксированный срез Amazon Reviews 2023 для категории Beauty and Personal Care и подтверждает, что канонический набор отзывов соответствует зарегистрированной версии данных. Канонический набор — это очищенное представление с единой схемой, на которое смогут опираться следующие этапы проекта.

## Цели

- проверить зарегистрированные исходные файлы и однозначно определить версию датасета;
- посмотреть реальные примеры отзывов и метаданных товаров;
- не путать наблюдения по небольшой выборке с точной статистикой по всем данным;
- проверить баланс строк после очистки, дедупликацию и каноническую схему;
- зафиксировать ограничения данных до построения каталога и ML-моделей.

## Inputs and outputs

Inputs are the versioned dataset manifest, immutable Amazon source files, the filtered working file, and the canonical review artifact produced by src.preprocessing.reviews.

Outputs are displayed audit results. Reusable calculations live in src/; this notebook does not implement ingestion or cleaning.

## Входы и результаты

На вход подаются манифест версии датасета, неизменяемые исходные файлы Amazon, ранее отфильтрованный рабочий файл и канонический Parquet-файл, созданный модулем src.preprocessing.reviews. Манифест хранит ожидаемые свойства файлов и не дает случайно смешать разные версии данных.

Результат ноутбука — наглядный отчет для проверки человеком. Переиспользуемые вычисления находятся в src/: ноутбук только вызывает их и объясняет результат, но не дублирует реализацию загрузки и очистки.

In [ ]:
# Standard library / Стандартная библиотека
import gzip
import json
import sys
from pathlib import Path

# Third-party packages / Сторонние библиотеки
import duckdb
import matplotlib.pyplot as plt
import pandas as pd
import pyarrow.parquet as pq
import seaborn as sns
from IPython.display import display

# Locate the repository root so local imports work from any notebook directory.
# Находим корень репозитория, чтобы локальные импорты работали независимо от
# директории, из которой был запущен Jupyter.
PROJECT_ROOT = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "PLAN.md").is_file() and (candidate / "src").is_dir()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Local project modules / Локальные модули проекта
from src.analytics.dataset_profile import profile_canonical_reviews
from src.common.project import find_project_root
from src.ingestion.dataset_manifest import (
    load_dataset_manifest,
    manifest_is_valid,
    verify_dataset_manifest,
)
from src.preprocessing.reviews import (
    CANONICAL_REVIEW_SCHEMA,
    validate_canonical_review_parquet,
)

In [ ]:
# Keep versioned paths and sample limits together so the run is reproducible.
# Храним версионные пути и размер выборки в одном месте, чтобы запуск можно
# было точно повторить и проверить.
PROJECT_ROOT = find_project_root(PROJECT_ROOT)
MANIFEST_PATH = (
    PROJECT_ROOT
    / "config/datasets/amazon_reviews_2023_beauty_2021_2023_v1.json"
)
QUALITY_REPORT_PATH = (
    PROJECT_ROOT
    / "reports/data_quality/amazon_reviews_2023_beauty_2021_2023_v1.json"
)
SAMPLE_SIZE = 10_000

# Resolve source paths through the manifest instead of hard-coding each file.
# Получаем пути исходных файлов из манифеста, а не прописываем их вручную:
# так смена версии датасета остается контролируемой.
manifest = load_dataset_manifest(MANIFEST_PATH)
canonical_registration = manifest.file_by_role("canonical_reviews")
CANONICAL_REVIEWS_PATH = PROJECT_ROOT / canonical_registration.path
RAW_REVIEWS_PATH = PROJECT_ROOT / manifest.file_by_role("raw_reviews").path
RAW_METADATA_PATH = (
    PROJECT_ROOT / manifest.file_by_role("raw_product_metadata").path
)
FILTERED_REVIEWS_PATH = (
    PROJECT_ROOT / manifest.file_by_role("filtered_reviews").path
)

# Display the active interpreter and critical dependency version so an
# environment mismatch is visible before expensive analysis starts.
# Показываем активный Python и версию важной зависимости, чтобы несовпадение
# окружений было видно до начала тяжелых вычислений.
print(f"Python executable: {sys.executable}")
print(f"DuckDB version: {duckdb.__version__}")
print(f"Dataset version: {manifest.dataset_version}")
print(f"Category: {manifest.dataset_category}")
print(f"Window: {manifest.date_window.start} through {manifest.date_window.end}")

## 1. Dataset manifest

The manifest fixes file identity, category, date window, sizes, checksums, and known row counts. Routine notebook execution checks file existence and size. Full checksum verification is available through the manifest CLI and is intentionally not repeated during every interactive run.

## Манифест датасета

Манифест фиксирует идентичность файлов, категорию Amazon, временной диапазон, размеры, контрольные суммы и известное количество строк. При обычном интерактивном запуске мы быстро проверяем наличие и размер файлов. Полный расчет SHA-256 доступен через CLI, но не повторяется каждый раз, потому что чтение многогигабайтных файлов заметно замедляет визуальную проверку.

In [ ]:
# Fast checks catch missing or replaced files without rereading every byte.
# Быстрые проверки находят отсутствующий или подмененный файл без полного
# повторного чтения нескольких гигабайт данных.
manifest_results = verify_dataset_manifest(
    manifest,
    project_root=PROJECT_ROOT,
    verify_checksums=False,
    verify_record_counts=False,
)
if not manifest_is_valid(manifest_results):
    raise ValueError("One or more dataset files do not match the manifest")

file_table = pd.DataFrame(manifest_results).merge(
    pd.DataFrame([item.model_dump() for item in manifest.files]),
    on=["role", "path"],
    how="left",
)
display(
    file_table[
        [
            "role",
            "path",
            "format",
            "compression",
            "record_count",
            "actual_size_bytes",
            "size_matches",
        ]
    ]
)

## 2. Source record inspection

The following cells read only five records from each compressed source. They are examples, not population statistics.

## Просмотр исходных записей

Следующие ячейки читают только по пять записей из каждого сжатого исходного файла. Это нужно, чтобы понять структуру полей и увидеть реальные значения. По пяти строкам нельзя делать выводы о всем датасете — точные статистики ниже рассчитываются по полной популяции.

In [ ]:
with gzip.open(RAW_REVIEWS_PATH, "rt", encoding="utf-8") as stream:
    raw_review_sample = pd.DataFrame(
        [json.loads(next(stream)) for _ in range(5)]
    )

review_schema_sample = pd.DataFrame(
    {
        "field": raw_review_sample.columns,
        "python_type": [
            type(raw_review_sample.iloc[0][column]).__name__
            for column in raw_review_sample.columns
        ],
        "missing_in_sample": [
            int(raw_review_sample[column].isna().sum())
            for column in raw_review_sample.columns
        ],
    }
)
display(review_schema_sample)
display(
    raw_review_sample[
        [
            "asin",
            "parent_asin",
            "rating",
            "title",
            "verified_purchase",
            "helpful_vote",
        ]
    ]
)

In [ ]:
with gzip.open(RAW_METADATA_PATH, "rt", encoding="utf-8") as stream:
    raw_metadata_sample = pd.DataFrame(
        [json.loads(next(stream)) for _ in range(5)]
    )

metadata_schema_sample = pd.DataFrame(
    {
        "field": raw_metadata_sample.columns,
        "python_type": [
            type(raw_metadata_sample.iloc[0][column]).__name__
            for column in raw_metadata_sample.columns
        ],
        "missing_in_sample": [
            int(raw_metadata_sample[column].isna().sum())
            for column in raw_metadata_sample.columns
        ],
    }
)
display(metadata_schema_sample)
display(
    raw_metadata_sample[
        [
            "parent_asin",
            "title",
            "main_category",
            "categories",
            "store",
            "average_rating",
            "rating_number",
        ]
    ]
)

## 3. Filtered working snapshot

This bounded sample verifies the filtered artifact structure and gives a visual preview. Exact counts come from the manifest and canonical population profile.

## Отфильтрованный рабочий срез

Ограниченная выборка проверяет структуру промежуточного файла и дает возможность визуально посмотреть отзывы. Ее размер специально ограничен, чтобы не загружать миллионы строк в память. Точное количество строк берется из манифеста и полного профиля канонического набора.

In [ ]:
filtered_sample = pd.read_json(
    FILTERED_REVIEWS_PATH,
    lines=True,
    nrows=SAMPLE_SIZE,
    convert_dates=False,
)
filtered_sample["review_timestamp"] = pd.to_datetime(
    filtered_sample["timestamp"], unit="ms", utc=True, errors="coerce"
)

sample_summary = pd.Series(
    {
        "sample_rows": len(filtered_sample),
        "sample_asins": filtered_sample["asin"].nunique(),
        "sample_parent_asins": filtered_sample["parent_asin"].nunique(),
        "sample_missing_text": filtered_sample["text"].isna().sum(),
        "sample_min_timestamp": filtered_sample["review_timestamp"].min(),
        "sample_max_timestamp": filtered_sample["review_timestamp"].max(),
    },
    name="value",
)
display(sample_summary.to_frame())
display(
    filtered_sample[["asin", "parent_asin", "rating", "title", "text"]].head()
)

## 4. Canonical build reconciliation

The preprocessing report must reconcile every input row into an accepted row or one primary rejection reason, then reconcile accepted rows into retained rows or exact duplicates.

## Сверка канонической сборки

Отчет очистки должен объяснить судьбу каждой входной строки. Сначала каждая строка либо проходит валидацию, либо получает одну основную причину отклонения. Затем все прошедшие строки делятся на сохраненные уникальные отзывы и удаленные точные дубликаты. Такая сверка защищает от незаметной потери данных.

In [ ]:
if not QUALITY_REPORT_PATH.is_file():
    raise FileNotFoundError(
        "Run the versioned preprocessing pipeline before this audit notebook"
    )
if not CANONICAL_REVIEWS_PATH.is_file():
    raise FileNotFoundError(CANONICAL_REVIEWS_PATH)

quality_report = json.loads(QUALITY_REPORT_PATH.read_text(encoding="utf-8"))
display(pd.Series(quality_report, name="value").to_frame())

# These equalities prove that no rows disappeared between processing stages.
# Эти равенства подтверждают, что между этапами обработки строки не исчезли
# без зафиксированной причины.
input_reconciles = quality_report["input_rows"] == (
    quality_report["valid_rows_before_deduplication"]
    + sum(quality_report["dropped_by_primary_reason"].values())
)
dedup_reconciles = quality_report["valid_rows_before_deduplication"] == (
    quality_report["output_rows"]
    + quality_report["duplicate_rows_removed"]
)
assert quality_report["dataset_version"] == manifest.dataset_version
assert quality_report["output_rows"] == canonical_registration.record_count
assert input_reconciles and dedup_reconciles
print(f"Input reconciliation passed: {input_reconciles}")
print(f"Deduplication reconciliation passed: {dedup_reconciles}")

In [ ]:
canonical_file = pq.ParquetFile(CANONICAL_REVIEWS_PATH)
canonical_validation = validate_canonical_review_parquet(
    CANONICAL_REVIEWS_PATH
)
assert canonical_validation.row_count == canonical_registration.record_count
assert canonical_validation.is_valid
print(f"Canonical rows from Parquet metadata: {canonical_file.metadata.num_rows:,}")
print(f"Row groups: {canonical_file.metadata.num_row_groups:,}")
display(
    pd.DataFrame(
        {
            "column": CANONICAL_REVIEW_SCHEMA.names,
            "type": [str(field.type) for field in canonical_file.schema_arrow],
            "expected_nullable": [
                field.nullable for field in CANONICAL_REVIEW_SCHEMA
            ],
            "physical_nullable": [
                field.nullable for field in canonical_file.schema_arrow
            ],
            "required_null_count": [
                canonical_validation.required_null_counts.get(field.name)
                for field in CANONICAL_REVIEW_SCHEMA
            ],
        }
    )
)

In [ ]:
# Unlike the previews above, this profile is calculated over every retained row.
# В отличие от примеров выше, этот профиль рассчитывается по всем сохраненным
# строкам и поэтому подходит для точных утверждений о датасете.
canonical_profile = profile_canonical_reviews(CANONICAL_REVIEWS_PATH)
population_summary = pd.Series(
    canonical_profile["summary"], name="value"
).to_frame()
rating_distribution = pd.DataFrame(canonical_profile["rating_distribution"])
year_distribution = pd.DataFrame(canonical_profile["year_distribution"])

display(population_summary)
display(rating_distribution)
display(year_distribution)

In [ ]:
sns.set_theme(style="whitegrid")
figure, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(
    data=rating_distribution,
    x="rating",
    y="review_count",
    color="#4C78A8",
    ax=axes[0],
)
axes[0].set_title("Exact Rating Distribution")
axes[0].set_xlabel("Rating")
axes[0].set_ylabel("Reviews")
sns.barplot(
    data=year_distribution,
    x="year",
    y="review_count",
    color="#72B7B2",
    ax=axes[1],
)
axes[1].set_title("Exact Review Volume by Year")
axes[1].set_xlabel("Year")
axes[1].set_ylabel("Reviews")
plt.tight_layout()
plt.show()

In [ ]:
canonical_preview = canonical_file.read_row_group(
    0,
    columns=[
        "review_id",
        "asin",
        "parent_asin",
        "rating",
        "review_title",
        "review_body",
        "review_timestamp",
        "verified_purchase",
        "helpful_vote",
    ],
).to_pandas()
display(canonical_preview.head(10))

## 5. Interpretation and limitations

### English

- The dataset is a fixed historical snapshot, not the current Amazon marketplace.
- Source reviews and product metadata are separate artifacts joined through `parent_asin`.
- Sample tables above support visual inspection only; population claims must come from the exact profile.
- Rating-derived sentiment groups are weak labels and are not textual sentiment predictions.
- Category-path completeness and review-to-metadata coverage are verified separately in notebooks 04 and 05.
- This audit passes only when both reconciliation checks pass and the displayed schema matches the canonical Amazon contract.

## Интерпретация и ограничения

### Русский

- Датасет является фиксированным историческим срезом и не отражает текущее состояние Amazon.
- Исходные отзывы и метаданные товаров хранятся отдельно и связываются через `parent_asin`.
- Таблицы с примерами нужны только для визуальной проверки; выводы обо всех данных должны опираться на полный точный профиль.
- Тональность, полученная из рейтинга, является слабой разметкой: это приближенная метка, а не предсказание тональности по тексту.
- Полнота путей категорий и покрытие соединения отзывов с товарами отдельно подтверждены в ноутбуках 04 и 05.
- Этот аудит считается успешным только тогда, когда обе сверки строк пройдены, а показанная схема совпадает с каноническим Amazon-контрактом.